# Smart ID Photo · v0.2 / Google Colab

Colab 是云环境，上传的照片会进入 Google 运行时；不是本机私密离线运行。默认只处理包内公开 NASA 样例，无自动分享链接、云视觉 API 或私照上传。此 notebook 尚未在实际 Colab 运行，只完成代码语法与本地引擎验证。

先运行上传项目 ZIP 单元，选择 Smart_ID_Photo_v0.2_LargeImages_15KB.zip。上传会传到 Colab。

In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import zipfile, stat, io
uploaded = files.upload()  # 主动上传项目包，不上传私人照片
if len(uploaded) != 1:
    raise ValueError("请选择一个项目 ZIP")
archive_bytes = next(iter(uploaded.values()))
base = Path("/content/smart-id-phase1").resolve()
base.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    entries = archive.infolist()
    if len(entries) > 1000 or sum(i.file_size for i in entries) > 200_000_000:
        raise ValueError("ZIP 太大")
    for item in entries:
        name = PurePosixPath(item.filename)
        if name.is_absolute() or ".." in name.parts or "\\" in item.filename or stat.S_ISLNK(item.external_attr >> 16):
            raise ValueError("不安全的 ZIP 路径")
        if not name.parts or name.parts[0] != "smart-id-photo":
            raise ValueError("不是预期项目包")
    archive.extractall(base)
project = base / "smart-id-photo"
assert (project / "pyproject.toml").is_file()
print(project)


In [ ]:
import subprocess, sys, os
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
# 仅依赖安装需要网络；不下载额外推理权重。
subprocess.run([sys.executable,"-m","pip","install","-e",str(project)+"[ui]"], check=True)
os.chdir(project)


In [ ]:
from smart_id_photo import __version__
print("项目版本：", __version__)
from smart_id_photo.processor import Processor
from smart_id_photo.config import Config
from IPython.display import display
from PIL import Image
result = Processor().process(project / "examples/astronaut.png", Config(output_dir=str(project/"output/colab")))
print(result.to_dict())
if result.preview: display(Image.open(result.preview))
if result.output: display(Image.open(result.output))


## 可选：交互界面
主动运行下一格会在 Colab 运行时开放端口视图。浏览器上传通过 Google/Colab 到运行时，不能任意读取你的本地文件夹。share=False，不创建 Gradio 公共隧道。若端口视图不支持，不要自动改为分享；使用上面的 Python 引擎或回到本机运行。

In [ ]:
from smart_id_photo.ui.app import build_app
from google.colab import output
app = build_app(str(project/"output/colab-ui"))
app.queue(max_size=2,default_concurrency_limit=1).launch(server_name="127.0.0.1", server_port=7860, share=False, inline=False, prevent_thread_lock=True, max_file_size="32mb", allowed_paths=[str(project/"output/colab-ui")])
output.serve_kernel_port_as_iframe(7860, height=900)


In [ ]:
if result.output:
    files.download(result.output)
# 使用完可调用 app.close()，并主动清理上传和输出文件。
